# Robustness and SSDC under RPI

This notebook aims to help you reproduce our key findings in subsections 5.3 (Disentangling Index-based and Content-based Spatial Organization) and 5.4 (Robustness to Content-preserving and Content-disrupting Spatial Organization).
In this notebook, you will load our pre-trained checkpoints (check repo README if you haven't downloaded them yet), evaluate their robustness to content-based distributional shifts, and analyse their SSDC under the RPI intervention.

**Memory note** : each model condition should be loaded and evaluated in a fresh session.
Running multiple conditions sequentially will OOM on most free-tier cloud hardware.

In [ ]:
# Importing needed dependencies

import sys
import os
sys.path.append(os.path.abspath("../"))
from src.main.model import VisionTransformer, ViTConfig
from src.metrics.robustness import evaluate_robustness_jpeg, evaluate_robustness_gaussian_blur
from src.metrics.ssdc import evaluate_ssdc
import torch


# Downloading Imagenet-100 dataset

from datasets import load_dataset

dataset_val = load_dataset("clane9/imagenet-100", split="validation")

In [ ]:
# Set which model you would like to test

condition = "APE" # Change to: "APE", "RoPE", "SPE", "RPT", or "ablated"
model_num = 1 # Must be an integer from 1 to 4

# Initializing model

DEFAULT_CONFIG = ViTConfig()
ViT = VisionTransformer(DEFAULT_CONFIG, condition)

# Loading checkpoint
path = os.path.abspath("../../models") + f"{condition}{model_num}_imagenet_100_60"
ViT.load_state_dict(torch.load(path)["model"])

In [ ]:
# Evaluate fragility scores of the model to two different forms of corruption

jpeg_score = evaluate_robustness_jpeg(ViT, dataset_val)
gaussian_blur_score = evaluate_robustness_gaussian_blur(ViT, dataset_val)

print(f"Fragility Score of model under JPEG corruption: {jpeg_score}")
print(f"Fragility Score of model under Gaussian Blur: {gaussian_blur_score}")

In [ ]:
# Evaluate and plot SSDC across depth of these models under RPI

ssdc_scores, cosine_maps = evaluate_ssdc(ViT, dataset_val, RPI = True)
layers = [i for i in range(12)]

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(layers, ssdc_scores, marker="o")
plt.xlabel("Layer")
plt.ylabel("SSDC")
plt.title("SSDC Across Depth Under RPI")
plt.grid(True)
plt.show()

## Visualising Token Similarity Structure

To better understand what SSDC is measuring, we visualise the mean token cosine similarity matrices from the first two layers.

We focus on these layers because they correspond approximately to the model state before and after the first encoder block, providing a good enough view of how spatial organisation emerges early in processing. Plotting all layers would substantially increase memory usage while providing limited additional intuition.

When examining these visualisations, look for:

- A diagonal band structure, indicating that nearby image patches tend to have similar representations.
- Grid-like spatial patterns, reflecting the model's awareness of relative spatial relationships between patches.
- Changes in these structures with and without the RPI intervention. (Turn RPI = False in previous cell if you'd like to make the comparison)

Models with positional encodings recover their spatial structure under RPI after the first encoder block (RoPE is the exception, as its spatial structure recovers more with more depth, not necessarily only after the first layer like SPE and APE). In contrast, models that rely on content-based organization (RPT and ablated models) do not exhibit this recovery. 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(cosine_maps[0])
axes[0].set_title("Layer 0")
axes[0].axis("off")

axes[1].imshow(cosine_maps[1])
axes[1].set_title("Layer 1")
axes[1].axis("off")

plt.tight_layout()
plt.show()